# Getting Started with OpenAgent

This notebook demonstrates how to create and use an OpenAgent agent.

OpenAgent is a general-purpose agent harness that gives LLM agents access to a computer via the terminal to complete tasks the way humans do.

We'll use **E2B cloud sandboxes** for secure, isolated execution.

## Helper Function for Streaming

We'll use streaming throughout this notebook to show responses in real-time.

Color-coded headers:
- **Yellow**: `<Human>` - User messages
- **Magenta**: `<AI (Thinking)>` - Assistant reasoning
- **Blue**: `<AI (Text)>` - Assistant responses
- **Green**: `<ToolCall: name>` - Tool invocations
- **Cyan**: `<ToolResult: name>` - Tool output

In [30]:
import json

from langchain_core.messages import AIMessageChunk, HumanMessage, ToolMessage
from langgraph.graph.state import CompiledStateGraph

# ANSI color codes
RESET = "\033[0m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
BLUE = "\033[34m"
MAGENTA = "\033[35m"
CYAN = "\033[36m"


def format_json(obj: dict | list | str) -> str:
    """Format object as readable JSON."""
    if isinstance(obj, str):
        return obj
    return json.dumps(obj, indent=2, ensure_ascii=False)


async def stream_agent(agent: CompiledStateGraph, message: str, config: dict | None = None) -> list:
    """Stream agent response with real-time output.

    Uses astream() with stream_mode="messages" for direct message streaming.
    Handles content_blocks properly:
    - "reasoning": Extended thinking/reasoning content
    - "text": Regular text content
    - "tool_use": Tool invocations
    - "tool_call_chunk": Streaming tool call chunks

    Returns:
        List of all (chunk, metadata) tuples streamed from the agent.
    """
    print(f"{YELLOW}<Human>{RESET}")
    print(message)
    print()

    inputs = {"messages": [HumanMessage(content=message)]}

    current_block = None  # "text", "reasoning", "tool_call", or None
    results = []

    async for chunk, metadata in agent.astream(inputs, config=config, stream_mode="messages"):
        results.append((chunk, metadata))

        # Handle ToolMessage (tool results)
        if isinstance(chunk, ToolMessage):
            if current_block:
                print("\n")
                current_block = None
            print(f"{CYAN}<ToolResult: {chunk.name}>{RESET}")
            print(str(chunk.content) if chunk.content else "(empty)")
            print()
            continue

        # Only process AI message chunks
        if not isinstance(chunk, AIMessageChunk):
            continue

        # Use content_blocks for proper streaming (preferred)
        content_blocks = getattr(chunk, "content_blocks", None)
        if content_blocks:
            for block in content_blocks:
                if not isinstance(block, dict):
                    continue

                block_type = block.get("type")

                if block_type == "reasoning":
                    reasoning = block.get("reasoning")
                    if reasoning:
                        if current_block != "reasoning":
                            if current_block:
                                print("\n")
                            print(f"{MAGENTA}<AI (Thinking)>{RESET}")
                            current_block = "reasoning"
                        print(reasoning, end="", flush=True)

                elif block_type == "text":
                    text = block.get("text")
                    if text:
                        if current_block != "text":
                            if current_block:
                                print("\n")
                            print(f"{BLUE}<AI (Text)>{RESET}")
                            current_block = "text"
                        print(text, end="", flush=True)

                elif block_type == "tool_use":
                    if current_block:
                        print("\n")
                    print(f"{GREEN}<ToolCall: {block.get('name', 'unknown')}>{RESET}")
                    print(format_json(block.get("input", {})))
                    print()
                    current_block = None

                elif block_type == "tool_call_chunk":
                    name = block.get("name")
                    args = block.get("args")
                    if name:
                        if current_block:
                            print("\n")
                        print(f"{GREEN}<ToolCall: {name}>{RESET}")
                        current_block = "tool_call"
                    if args:
                        print(args, end="", flush=True)

            continue

        # Fallback: handle content directly (for models without content_blocks)
        content = chunk.content
        if isinstance(content, str) and content:
            if current_block != "text":
                if current_block:
                    print("\n")
                print(f"{BLUE}<AI (Text)>{RESET}")
                current_block = "text"
            print(content, end="", flush=True)

    # End any open block
    if current_block:
        print("\n")

    return results

## Setting Up the Computer

OpenAgent needs a `Computer` to execute commands. Choose one:

- **`RemoteE2BComputer`** - Cloud sandbox via [E2B](https://e2b.dev). Isolated, safe, persistent.
- **`LocalNativeComputer`** - Your local machine. Use with caution.

We'll use E2B for this notebook.

In [31]:
from openagent.computer import RemoteE2BComputer

# Create a cloud sandbox with 3 minute lifetime
# Template "openagent" provides 2 CPU, 4 GiB memory ($0.17/hr)
computer = RemoteE2BComputer(template="openagent", lifetime=180)
await computer.start()

print(f"Sandbox ID: {computer.sandbox_id}")
print("(Save this ID to reconnect later)")

Sandbox ID: i0g0f893ekbu0msu6l4ps
(Save this ID to reconnect later)


## Creating the Agent

`create_agent()` is the main entry point. It wires together:
- **CLI tools** (bash, read, write, edit, glob, grep) bound to the computer
- **System prompt** assembled from modular prompt fragments
- **Runtime modules** (context compaction, permission gate, capability registry)

The `model` parameter accepts either a `BaseChatModel` instance or a model string (e.g. `"anthropic:claude-sonnet-4-20250514"`).

In [32]:
from openagent import create_agent

# Option A: Use a model string (uses langchain init_chat_model)
# agent = create_agent(computer, model="anthropic:claude-sonnet-4-20250514")

# Option B: Use a BaseChatModel instance directly
import os

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="kimi-k2.5",
    api_key=os.environ["UNICOM_API_KEY"],
    api_base=os.environ["UNICOM_BASE_URL"],
)

agent = create_agent(computer, model=model)

## Single-Turn Usage

The agent receives a task, reasons about it, and uses tools autonomously to complete it.

In [29]:
# Stream agent response in real-time
_ = await stream_agent(
    agent,
    "Check what OS this is. What system tools are available?",
)

<Human>
Check what OS this is. What system tools are available?

<AI (Thinking)>
 The user wants to know what operating system this is and what system tools are available. I'll start by checking the OS information and then list available system tools.

<ToolCall: bash>
{"command": "cat /etc/os-release 2>/dev/null || uname -a"}

<AI (Text)>
 

<ToolCall: bash>
{"command": "which bash sh python python3 node npm gcc make curl wget git 2>/dev/null | head -20"}

<AI (Text)>
  

<ToolResult: bash>
PRETTY_NAME="Ubuntu 24.04.3 LTS"
NAME="Ubuntu"
VERSION_ID="24.04"
VERSION="24.04.3 LTS (Noble Numbat)"
VERSION_CODENAME=noble
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=noble
LOGO=ubuntu-logo

<ToolResult: bash>
/usr/bin/bash
/usr/bin/sh
/usr/bin/python3
/usr/bin/node
/usr/bin/npm
/usr/bin/gcc


## Adding Web Search and Fetch

Pass `search_provider` and `fetch_provider` to give the agent web access. Available providers:

| Capability | Providers |
|---|---|
| **Search** | `TavilySearchProvider` (requires `TAVILY_API_KEY`), `BraveSearchProvider` |
| **Fetch** | `JinaFetchProvider` (works without key), `FirecrawlFetchProvider` |

In [33]:
from openagent.tools.web import JinaFetchProvider, TavilySearchProvider

# Create an agent with web capabilities
# TavilySearchProvider reads TAVILY_API_KEY from env by default
# JinaFetchProvider works without a key (optional JINA_API_KEY for higher limits)
web_agent = create_agent(
    computer,
    model=model,
    search_provider=TavilySearchProvider(),
    fetch_provider=JinaFetchProvider(),
)

TypeError: create_agent() got an unexpected keyword argument 'search_provider'

In [ ]:
# The agent can now search the web and fetch page content
_ = await stream_agent(
    web_agent,
    "Search for 'Python 3.13 new features'. Summarize the top results.",
)

In [ ]:
# web_fetch extracts clean text from any URL
_ = await stream_agent(
    web_agent,
    "Fetch https://docs.python.org/3/whatsnew/3.13.html and list the 5 most important changes.",
)

## Multi-Turn Conversations

Pass a `checkpointer` to maintain state across multiple invocations. Each "thread" gets its own conversation history.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Create agent with checkpointer for persistent memory
checkpointer = MemorySaver()
agent = create_agent(computer, model=model, checkpointer=checkpointer)

# Session config for multi-turn
config = {"configurable": {"thread_id": "my-session"}}

In [ ]:
# First turn - install a package (persists in sandbox)
_ = await stream_agent(
    agent,
    "Install the 'cowsay' package with pip (using --break-system-packages)",
    config=config,
)

In [ ]:
# Second turn - use the installed package (agent remembers context)
_ = await stream_agent(
    agent,
    "Use cowsay to say 'Hello from E2B!'",
    config=config,
)

In [ ]:
# Third turn - agent remembers what we did
_ = await stream_agent(
    agent,
    "What packages did we install in this session?",
    config=config,
)

## Cleanup

Stop the sandbox when done. This pauses it (preserving state) rather than destroying it.

In [ ]:
await computer.stop()
print("Sandbox paused. State preserved for reconnection.")